# Generating and morphing Fashion-MNIST items with bidirectional predictive coding

**Goal.** Train a bidirectional predictive coding (bPC) network on Fashion-MNIST and use it as a
generator: clamp a class label, leave the image free, and let PC inference shape the image into an
item of that class. Then swap the clamped label mid-run and watch the item morph into another class.

**Dataset.** Fashion-MNIST: 28×28 grayscale images of clothing, 60,000 for training and 10,000 for
testing, in 10 classes:

| label | class | label | class |
|---|---|---|---|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

**Sections**

1. Setup
2. Data
3. Model
4. Training
5. Saving and reloading the weights
6. Generation as inference
7. Snapshots
8. When does the image change class?
9. GIF of all ten classes
10. One image through a sequence of classes

## 1. Setup

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from tqdm import tqdm

import predictive_coding as pc

# Reproducibility: same random numbers every run
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Paths are relative to the notebook, which lives in RELAX/notebooks/.
# data/ and runs/ are both listed in .gitignore, so nothing here gets committed.
DATA_DIR = Path("../data")
RUN_DIR = Path("../runs/fashion_mnist_bpc_generation")
RUN_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")

# Class names, indexed by label
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## 2. Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                          # PIL image -> tensor in [0, 1], shape (1, 28, 28)
    transforms.Lambda(lambda x: torch.flatten(x)),  # -> shape (784,)
])

train_dataset = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)

batch_size = 500
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(len(train_dataset), len(test_dataset))

## 3. Model

Two pathways share the same two latent layers $x_0, x_1$ (the `PCLayer`s):

- **bottom-up (recognition):** image $u$ → $x_0$ → $x_1$ → label $\hat y$
- **top-down (generation):** label $y$ → $x_1$ → $x_0$ → image $\hat u$

The energy is the sum of all squared prediction errors of both pathways, weighted by
$\alpha_\uparrow$ and $\alpha_\downarrow$. Training, classification and generation all use this one
energy; they differ only in which variables are clamped and which are free.

In [ ]:
class BidirectionalPC(nn.Module):
    def __init__(self, alpha_up=1.0, alpha_down=1.0):
        super().__init__()
        # The two latent layers x0 and x1. Each is used by BOTH pathways.
        self.pcs = nn.ModuleList([pc.PCLayer(), pc.PCLayer()])
        act = nn.ReLU()

        # bottom-up: image -> x0 -> x1 -> label
        self.model_up = nn.Sequential(
            nn.Linear(784, 256),
            self.pcs[0],
            act,
            nn.Linear(256, 256),
            self.pcs[1],
            act,
            nn.Linear(256, 10),
        )
        # top-down: label -> x1 -> x0 -> image   (note: pcs in reverse order)
        self.model_down = nn.Sequential(
            nn.Linear(10, 256),
            self.pcs[1],
            act,
            nn.Linear(256, 256),
            self.pcs[0],
            act,
            nn.Linear(256, 784),
        )

        self.alpha_up = alpha_up
        self.alpha_down = alpha_down

    def forward(self, inputs):
        img, y = inputs
        # 1) Bottom-up pass. Each PCLayer stores the energy 0.5*(x - mu_up)^2 in ._energy
        out_up = self.model_up(img)
        e_up = [p._energy for p in self.pcs]
        # 2) Top-down pass. It OVERWRITES ._energy with 0.5*(x - mu_down)^2
        out_down = self.model_down(y)
        e_down = [p._energy for p in self.pcs]
        # 3) Store the weighted sum, which is what PCTrainer will read
        for p, eu, ed in zip(self.pcs, e_up, e_down):
            p._energy = self.alpha_up * eu + self.alpha_down * ed
        return out_up, out_down


def loss_fn(outputs, _y, _img, alpha_up, alpha_down):
    """Energy of the two *output* terms: 0.5||y - y_hat||^2 and 0.5||u - u_hat||^2."""
    out_up, out_down = outputs
    return (alpha_up * 0.5 * (out_up - _y).pow(2).sum()
            + alpha_down * 0.5 * (out_down - _img).pow(2).sum())

In [ ]:
alpha_up, alpha_down = 1.0, 1.0
model = BidirectionalPC(alpha_up, alpha_down).to(device)
print(model)
print("weights:", sum(p.numel() for p in model.parameters()))

## 4. Training

For every batch, `PCTrainer.train_on_batch` clamps the image and the label, initialises the latents
with a forward pass, runs `T` inference steps on the latents (SGD), and then takes one weight step
(Adam). Accuracy is measured with a single bottom-up feed-forward pass.

In [ ]:
T = 20  # inference steps per batch

trainer = pc.PCTrainer(
    model,
    T=T,
    optimizer_x_fn=optim.SGD, optimizer_x_kwargs={"lr": 0.01},   # inference
    update_p_at="last",
    optimizer_p_fn=optim.Adam, optimizer_p_kwargs={"lr": 1e-3},  # learning
    plot_progress_at=[],
)


def test_accuracy(model):
    """Classify with one bottom-up feed-forward pass (PCLayers transparent in eval mode)."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for img, label in test_loader:
            pred = model.model_up(img.to(device)).argmax(dim=1)
            correct += (pred == label.to(device)).sum().item()
    model.train()
    return correct / len(test_dataset)

In [ ]:
epochs = 5
model.train()  # PCLayers must be in train mode while training (the trainer asserts this)

accs = [test_accuracy(model)]
for epoch in range(epochs):
    for img, label in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        img = img.to(device)
        y = F.one_hot(label, num_classes=10).float().to(device)
        trainer.train_on_batch(
            inputs=(img, y),
            loss_fn=loss_fn,
            loss_fn_kwargs={"_y": y, "_img": img, "alpha_up": alpha_up, "alpha_down": alpha_down},
            is_log_progress=False,
            is_return_results_every_t=False,
            is_checking_after_callback_after_t=False,
        )
    accs.append(test_accuracy(model))
    print(f"epoch {epoch + 1}: test accuracy {accs[-1]:.4f}")

plt.plot(accs, marker="o")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.grid(alpha=0.3)
plt.show()

## 5. Saving and reloading the weights

`PCLayer` stores its latent activity `_x` as an `nn.Parameter`, so it ends up in `state_dict()`
with the shape of the last training batch. It is not a weight, so we drop it before saving.
Generation then uses a fresh model loaded from the checkpoint, with frozen weights.

In [ ]:
def weights_only(state_dict):
    """Drop the PCLayer latents (keys ending in '._x') from a state_dict."""
    return {k: v for k, v in state_dict.items() if not k.endswith("._x")}

ckpt_path = RUN_DIR / f"bpc_up{alpha_up}_down{alpha_down}.pt"
torch.save(weights_only(model.state_dict()), ckpt_path)
print("saved", ckpt_path)

In [ ]:
gen_model = BidirectionalPC(alpha_up, alpha_down).to(device)
print(gen_model.load_state_dict(torch.load(ckpt_path)))

# Freeze the weights: during generation only the latents and the image change.
for p in gen_model.parameters():
    p.requires_grad_(False)

## 6. Generation as inference

The label $y$ is clamped; the latents $x_0, x_1$ **and the image $u$** are free and follow gradient
descent on the energy. Changing the clamped label between phases of the `schedule` is the *swap*.

Two independent choices set the starting point:

| `image_init` | the image starts as… |
|---|---|
| `"topdown"` | the one-pass top-down picture of the first label |
| `"noise"` | uniform random pixels |

| `latent_init` | the latents start as… |
|---|---|
| `"topdown"` | top-down predictions from the label (they already "know" the class) |
| `"bottomup"` | bottom-up predictions from the starting image (only the clamped label knows the class) |

`image_init="noise"` with `latent_init="bottomup"` is true generation from noise.
The inference step `lr_x` must stay below the stability limit $2/\lambda_{\max}$ of the energy's
Hessian; `0.02` is safe for this model.

In [ ]:
def one_hot(labels):
    return F.one_hot(torch.as_tensor(labels), num_classes=10).float().to(device)


def generate(model, schedule, lr_x=0.02, image_init="topdown", latent_init="topdown",
             record_every=40):
    """
    Run PC inference with the label clamped and the image free.

    schedule:    list of (labels, n_steps), one entry per phase; e.g.
                 [([0, 1, ..., 9], 300), ([1, 2, ..., 0], 10000)] generates ten images at once
                 and swaps each one to the next class after 300 steps
    image_init:  "topdown"  -> image starts as the top-down picture of the first label
                 "noise"    -> image starts as uniform noise
    latent_init: "topdown"  -> latents start as top-down predictions from the first label
                 "bottomup" -> latents start as bottom-up predictions from the starting image
    Returns frames (list of (batch, 784) tensors), the clamped labels of each frame,
    and the energy at every step.
    """
    model.train()  # PCLayers must use their latents _x, not be transparent
    y0 = one_hot(schedule[0][0])
    batch = y0.shape[0]

    # (a) Where the IMAGE starts. It becomes a free variable: an nn.Parameter
    #     that the optimiser can update.
    if image_init == "topdown":
        model.eval()                          # transparent PCLayers: a plain top-down pass
        with torch.no_grad():
            img0 = model.model_down(y0)       # one-pass picture of the first label
        model.train()
    elif image_init == "noise":
        img0 = torch.rand(batch, 784, device=device)
    else:
        raise ValueError(f"unknown image_init: {image_init}")
    img = nn.Parameter(img0.clone())

    # (b) Where the LATENTS start: ONE forward pass with sampling switched on.
    #     set_is_sample_x(True) makes each PCLayer set _x := mu on the next forward call.
    #     This comes after (a) because bottom-up latents are computed FROM the image.
    for p in model.pcs:
        p.set_is_sample_x(True)
    if latent_init == "topdown":
        model.model_down(y0)                  # x1, x0 := predictions from the label
    elif latent_init == "bottomup":
        model.model_up(img)                   # x0, x1 := predictions from the starting image
    else:
        raise ValueError(f"unknown latent_init: {latent_init}")

    # (c) Inference optimiser: plain gradient descent on x0, x1 and u. The weights are frozen.
    opt = optim.SGD([p._x for p in model.pcs] + [img], lr=lr_x)

    frames, frame_labels, energies = [img.detach().clone()], [schedule[0][0]], []
    step = 0
    for labels, n_steps in schedule:
        y = one_hot(labels)  # <- the clamp. Changing this is the "swap".
        for _ in range(n_steps):
            opt.zero_grad()
            out_up, out_down = model((img, y))
            E = sum(p._energy for p in model.pcs) + loss_fn(
                (out_up, out_down), y, img, model.alpha_up, model.alpha_down)
            E.backward()   # dE/dx0, dE/dx1, dE/du
            opt.step()     # x <- x - lr * dE/dx
            energies.append(E.item())
            step += 1
            if step % record_every == 0:
                frames.append(img.detach().clone())
                frame_labels.append(labels)
    return frames, frame_labels, energies

All ten classes at once (one image per class, each row of the batch is independent). After 300
steps every item is swapped to the next class: T-shirt/top → Trouser, Trouser → Pullover, …,
Ankle boot → T-shirt/top.

In [ ]:
record_every = 40
A = list(range(10))
B = [(c + 1) % 10 for c in A]
schedule = [(A, 300), (B, 30_000)]

frames, frame_labels, energies = generate(gen_model, schedule, lr_x=0.02, record_every=record_every)
print(len(frames), "frames")

plt.semilogy(energies)
plt.axvline(schedule[0][1], color="r", ls="--", label="label swap")
plt.xlabel("inference step"); plt.ylabel("energy E"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7. Snapshots

One column per item, one row per moment in time (rows are denser right after the swap).

In [ ]:
def show_snapshots(frames, frame_labels, record_every, n_rows=8):
    idx = np.unique(np.geomspace(1, len(frames) - 1, n_rows - 1).astype(int))
    idx = [0] + list(idx)
    fig, axes = plt.subplots(len(idx), 10, figsize=(12, 1.2 * len(idx)))
    for r, k in enumerate(idx):
        for c in range(10):
            ax = axes[r, c]
            ax.imshow(frames[k][c].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{CLASSES[frame_labels[0][c]]}\n→ {CLASSES[frame_labels[-1][c]]}",
                             fontsize=7)
        axes[r, 0].set_ylabel(f"t={k * record_every}", fontsize=7)
    plt.tight_layout(); plt.show()

show_snapshots(frames, frame_labels, record_every)

## 8. When does the image change class?

The bottom-up pathway in eval mode is a plain feed-forward classifier. Applied to the stored
frames, it tells us which class each image looks like at each moment.

In [ ]:
def classify(model, images):
    """Feed-forward classification of a (batch, 784) tensor; returns a list of labels."""
    model.eval()   # transparent PCLayers -> feed-forward classifier
    with torch.no_grad():
        pred = model.model_up(images).argmax(1).tolist()
    model.train()  # back to latent mode
    return pred

before = classify(gen_model, frames[schedule[0][1] // record_every])  # last frame before the swap
after = classify(gen_model, frames[-1])
print("before swap:", [CLASSES[c] for c in before])
print("after  swap:", [CLASSES[c] for c in after])

In [ ]:
# Print the predicted classes every 10 frames and flag the moments where any of them changes
prev = None
for i in range(0, len(frames), 10):
    pred = classify(gen_model, frames[i])
    flag = "  <- changed" if prev is not None and pred != prev else ""
    print(f"step {i * record_every:6d}: {pred}{flag}")
    prev = pred

## 9. GIF of all ten classes

Each panel is titled with the class currently clamped on it.

In [ ]:
def save_gif(frames, frame_labels, path, record_every, fps=20):
    fig, axes = plt.subplots(1, 10, figsize=(12, 1.9))
    ims, titles = [], []
    for c, ax in enumerate(axes):
        ims.append(ax.imshow(frames[0][c].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1))
        titles.append(ax.set_title("", fontsize=7))
        ax.axis("off")
    suptitle = fig.suptitle("")

    def update(k):
        for c in range(10):
            ims[c].set_data(frames[k][c].view(28, 28).clamp(0, 1))
            titles[c].set_text(CLASSES[frame_labels[k][c]])
        suptitle.set_text(f"step {k * record_every}")
        return ims

    FuncAnimation(fig, update, frames=len(frames)).save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)

gif_path = RUN_DIR / "morph_all_classes.gif"
save_gif(frames, frame_labels, gif_path, record_every)
print("saved", gif_path)

## 10. One image through a sequence of classes

Generate **one** image and clamp a list of classes on it, one after the other: e.g. `[1, 8, 4]`
clamps Trouser, then Bag, then Coat. The GIF shows the image next to the energy curve; dashed lines
mark each clamp switch and the orange cursor is the current step.

In [ ]:
def make_sequence_gif(model, sequence, steps_per_clamp, lr_x=0.02,
                      image_init="topdown", latent_init="topdown",
                      record_every=20, fps=25, path=None):
    """
    Generate ONE image and clamp a sequence of class labels on it, one after the other.

    sequence:        class labels to clamp in order, e.g. [1, 8, 4] = Trouser -> Bag -> Coat
    steps_per_clamp: inference steps for each clamp, either one int (same for all)
                     or a list with one entry per class, e.g. [300, 5000, 5000]
    image_init, latent_init: passed to generate (see its docstring)
    Saves a GIF with the image on the left and the energy on the right, and
    returns (frames, frame_labels, energies, path).
    """
    if isinstance(steps_per_clamp, int):
        steps_per_clamp = [steps_per_clamp] * len(sequence)
    assert len(steps_per_clamp) == len(sequence), "need one step count per class"

    # A batch of ONE: every phase clamps a list with a single label, e.g. ([8], 5000)
    schedule = [([c], n) for c, n in zip(sequence, steps_per_clamp)]
    frames, frame_labels, energies = generate(model, schedule, lr_x=lr_x,
                                              image_init=image_init, latent_init=latent_init,
                                              record_every=record_every)

    # step at which each new clamp starts, to mark the switches on the energy plot
    switch_steps = np.cumsum([0] + steps_per_clamp)[1:-1]

    fig, (ax_img, ax_E) = plt.subplots(1, 2, figsize=(8, 3.6),
                                       gridspec_kw={"width_ratios": [1, 1.6]})
    im = ax_img.imshow(frames[0][0].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax_img.axis("off")

    ax_E.semilogy(energies, color="#2a78d6", lw=1)
    for s in switch_steps:
        ax_E.axvline(s, color="#52514e", ls="--", lw=1)
    cursor = ax_E.axvline(0, color="#eb6834", lw=2)       # moves with the animation
    ax_E.set_xlabel("inference step"); ax_E.set_ylabel("energy E")
    ax_E.set_title(f"image init: {image_init}   latent init: {latent_init}", fontsize=8)
    ax_E.grid(alpha=0.3)
    title = fig.suptitle("")
    fig.tight_layout()

    def update(k):
        step = k * record_every
        im.set_data(frames[k][0].view(28, 28).clamp(0, 1))
        cursor.set_xdata([step, step])
        title.set_text(f"clamped class: {CLASSES[frame_labels[k][0]]}    step {step}")
        return im, cursor, title

    if path is None:
        # include the init choices so different runs don't overwrite each other
        path = RUN_DIR / (f"sequence_{'-'.join(map(str, sequence))}"
                          f"_img-{image_init}_lat-{latent_init}.gif")
    FuncAnimation(fig, update, frames=len(frames)).save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return frames, frame_labels, energies, path

In [ ]:
sequence = [1, 8, 4]                   # class labels to clamp in order
steps_per_clamp = [300, 10000, 10000]  # steps for each clamp (or a single int for all)
image_init = "topdown"                 # "topdown" or "noise"
latent_init = "topdown"                # "topdown" or "bottomup"
# For true generation from noise use image_init="noise", latent_init="bottomup",
# and give the first clamp ~6000 steps.

print(" -> ".join(CLASSES[c] for c in sequence))
seq_frames, seq_labels, seq_energies, seq_gif = make_sequence_gif(
    gen_model, sequence, steps_per_clamp, lr_x=0.02,
    image_init=image_init, latent_init=latent_init, record_every=20, fps=25)

print(len(seq_frames), "frames ->", seq_gif)
display(Image(filename=str(seq_gif)))   # shows the animated GIF inline